# Nhánh P1 trên Colab: kiểm mẫu tự chọn bằng CLIP trước lượt 2

Chạy **lần lượt từ trên xuống**. Ô nào báo lỗi thì dừng lại, chụp lỗi gửi lại, đừng chạy tiếp.

| Ô | Làm gì | Tốn bao lâu (ước lượng) |
|---|---|---|
| 1–2 | Kiểm GPU, mount Google Drive | 1 phút |
| 3 | Lấy code CS331 từ GitHub | 1 phút |
| 4 | Tạo môi trường riêng Python 3.10 giống máy Trí + CUDA ops. **Lần đầu tiên** phải tải CUDA 12.1 và build (10–15 phút), file build lưu vào Drive; **các phiên sau** chỉ cài lại (3–5 phút). Không cần Restart | 3–15 phút |
| 5–7 | Kiểm môi trường, tải ảnh MUCCA (Zenodo) và checkpoint CountGD++ | 3–5 phút |
| 8 | **Kiểm A:** CLIP trên Colab cho điểm giống bản đã lưu không | 1–2 phút |
| 9 | **Kiểm B + chạy P1** trên 5 ảnh pilot (15 query): so B0/B2 với bản đã lưu (A2000), rồi thử 5 ngưỡng | 2–10 phút |
| 10 | Bảng tóm tắt (dán kết quả ô này gửi lại) | vài giây |
| 11 | **Kiểm C:** lần chạy này có khớp tuyệt đối một lần chạy T4 trước không + mẫu B2 có trùng A2000 không | vài giây |
| 12 | **Bảng đối chứng** (kế hoạch Trí §3.5): F, P+F, B0+F (C6), S-only từ một lần chạy P1, không chạy lại CountGD++. Phiên mới chỉ cần Ô 1–5 trước | 3–5 phút |

**Máy:** dùng **T4** (Runtime > Change runtime type > T4 GPU). CountGD++ chỉ cần vài GB VRAM (Trí từng chạy trên GPU laptop 4 GB); T4 tốn khoảng 2 compute unit/giờ, A100 khoảng 11–13. Xong việc: **Runtime > Disconnect and delete runtime** (để máy chờ vẫn bị trừ unit).

**Đã biết (03/11):** trên T4, Kiểm B không khớp tuyệt đối với bản A2000 (lượt 1 lệch tối đa 1 khung, do khác GPU), còn Kiểm C đạt (chạy lại trên T4 khớp tuyệt đối 6 cách × 15/15). Vì vậy chỉ so các cách **trong cùng một lần chạy T4**, không trộn số T4 với số A2000. Tiêu chí mới chờ nhóm chốt; xem `COLAB_P1_HUONG_DAN.md`.

**Quy tắc (kế hoạch của Trí §3.2, §5):** 5 ảnh pilot chỉ để **dò lỗi và kiểm cài đặt**. Ngưỡng `tau_pseudo` và `M` chính thức phải chọn trên **tập dev 30 ảnh mới** sau khi nhóm khóa danh sách. **Không** chạy để chỉnh ngưỡng trên `p4_unseen_v1` (10 ảnh đã xem). Model không bao giờ đọc GT; GT chỉ dùng ở ô 10 để tính MAE.

In [ ]:
# Ô 1: GPU. Nên thấy Tesla T4 (Runtime > Change runtime type > T4 GPU). A100/L4 cũng chạy được nhưng tốn unit hơn.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Ô 2: Google Drive để lưu kết quả và checkpoint (khỏi tải lại lần sau).
# Nếu không muốn dùng Drive (hoặc mount lỗi): đổi USE_DRIVE = False, kết quả nằm ở /content/CS331_colab_out.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Ô 3: lấy code. REPO_REF = "main" là bản mới nhất; muốn chạy lại đúng một bản cũ thì ghi mã commit vào đây.
REPO_REF = "main"
!test -d /content/CS331 || git clone -q https://github.com/minhtridaoo/CS331.git /content/CS331
!cd /content/CS331 && git fetch -q origin && git checkout -q {REPO_REF} && (git pull -q --ff-only 2>/dev/null || true)
!cd /content/CS331 && git log -1 --format="commit %H %ad %s" && test -f experiments/countgdpp/run_b2_verified_pseudo.py && echo "Có runner P1"

In [ ]:
# Ô 4: môi trường riêng /content/venv310 giống máy Trí (Python 3.10, torch 2.5.1+cu121, transformers 4.44.2,
# numpy 2.2.6, pillow 10.4.0). Colab mặc định Python 3.13 không có torch 2.5.1, nên KHÔNG cài vào kernel của notebook;
# CountGD++ và CLIP sẽ chạy bằng Python của venv này. Không cần Restart session.
# Build CUDA ops ở bản sao /content/ops_build để không đụng file trong repo. Chỉ chạy lần đầu của mỗi phiên Colab.
# Colab có sẵn CUDA 13 + gcc 13, còn torch 2.5.1 build bằng CUDA 12.1, nên tải bộ CUDA 12.1 chính thức của NVIDIA
# (~4 GB, chỉ phần toolkit) vào /content/cuda-12.1 và dùng gcc-12 khi build. CUDA 13 của Colab giữ nguyên.
# File build (wheel) dùng được cho T4, A100, L4 và được lưu ở MyDrive/CS331_colab/wheels; phiên sau cài thẳng từ đó.
import glob, os, shutil, subprocess
PY = "/content/venv310/bin/python"
CUDA121 = "/content/cuda-12.1"
WHEELS = "/content/drive/MyDrive/CS331_colab/wheels"
ARCH = "7.5;8.0;8.6;8.9"  # T4 = 7.5, A100 = 8.0, A2000 = 8.6, L4 = 8.9
def sh(cmd):
    # In từng dòng output của lệnh (subprocess.run thường không hiện output trong Colab).
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"Lệnh lỗi (mã {p.returncode}), xem các dòng ngay phía trên.")
def find_wheel(folder):
    # setuptools mới viết tên file chữ thường (multiscaledeformableattention-...), bản cũ viết MultiScale...
    return sorted(p for p in glob.glob(f"{folder}/*.whl")
                  if os.path.basename(p).lower().startswith("multiscaledeformableattention-1.0-cp310"))
if os.path.exists("/content/.cs331_venv_ok"):
    print("Đã cài trong phiên này, bỏ qua. Chạy tiếp Ô 5.")
else:
    sh("pip install -q uv")
    if not os.path.exists(PY):
        sh("uv venv /content/venv310 --python 3.10")
    sh(f"uv pip install -p {PY} torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121")
    sh(f"uv pip install -p {PY} transformers==4.44.2 timm==1.0.30 numpy==2.2.6 pillow==10.4.0 scipy termcolor addict yapf==0.40.1"
       " opencv-python-headless pycocotools matplotlib setuptools wheel ninja")
    cached = find_wheel(WHEELS)
    if cached:
        print("Dùng file build đã lưu trên Drive:", cached[-1])
        sh(f"uv pip install -p {PY} --reinstall {cached[-1]}")
    else:
        if not os.path.exists(f"{CUDA121}/bin/nvcc"):
            sh("wget -q -O /content/cuda121.run https://developer.download.nvidia.com/compute/cuda/12.1.1/local_installers/cuda_12.1.1_530.30.02_linux.run")
            sh(f"sh /content/cuda121.run --silent --toolkit --toolkitpath={CUDA121} --override --no-man-page")
            sh("rm -f /content/cuda121.run")
        sh(f"{CUDA121}/bin/nvcc --version | tail -1")
        sh("apt-get -qq update && apt-get -qq install -y gcc-12 g++-12")
        sh("rm -rf /content/ops_build /content/wheels && cp -r /content/CS331/CountGDPlusPlus/models/GroundingDINO/ops /content/ops_build")
        sh("rm -rf /content/ops_build/build /content/ops_build/dist /content/ops_build/*.egg-info")
        sh(f"cd /content/ops_build && CUDA_HOME={CUDA121} PATH={CUDA121}/bin:/content/venv310/bin:$PATH CC=gcc-12 CXX=g++-12"
           f" TORCH_CUDA_ARCH_LIST='{ARCH}' {PY} setup.py -q bdist_wheel -d /content/wheels")
        built = find_wheel("/content/wheels")
        if not built:
            raise RuntimeError(f"Không thấy file build trong /content/wheels: {os.listdir('/content/wheels')}")
        wheel = built[-1]
        sh(f"uv pip install -p {PY} --reinstall {wheel}")
        if os.path.isdir("/content/drive/MyDrive"):
            os.makedirs(WHEELS, exist_ok=True)
            shutil.copy(wheel, WHEELS)
            print("Đã lưu file build vào", WHEELS)
    open("/content/.cs331_venv_ok", "w").write("ok")
    print("CÀI XONG. Chạy tiếp Ô 5.")

In [ ]:
# Ô 5: đường dẫn + kiểm môi trường venv (nếu lỡ Restart session thì chạy lại từ ô này; file vẫn còn).
import datetime, hashlib, json, os, pathlib, shlex, shutil, subprocess, sys
ROOT = pathlib.Path("/content/CS331")
PY = "/content/venv310/bin/python"
OUT_ROOT = pathlib.Path("/content/drive/MyDrive/CS331_colab" if os.path.isdir("/content/drive/MyDrive") else "/content/CS331_colab_out")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["MPLBACKEND"] = "Agg"  # Colab đặt backend inline; matplotlib của venv không có nên phải đổi cho các lệnh con
def sh(cmd):
    # In từng dòng output của lệnh (subprocess.run thường không hiện output trong Colab).
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"Lệnh lỗi (mã {p.returncode}), xem các dòng ngay phía trên.")
def file_hash(path, algo="sha256"):
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()
probe = ("import sys, torch, transformers, numpy, PIL, MultiScaleDeformableAttention; "
         "assert torch.__version__.startswith('2.5.1'), torch.__version__; "
         "print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__, "
         "'| numpy', numpy.__version__, '| pillow', PIL.__version__, '| GPU', torch.cuda.get_device_name(), "
         "'| TF32 matmul', torch.backends.cuda.matmul.allow_tf32, '| TF32 cudnn', torch.backends.cudnn.allow_tf32)")
sh(f"{PY} -c {shlex.quote(probe)}")  # lỗi ở đây = Ô 4 chưa xong (MultiScaleDeformableAttention = CUDA ops)
print("Kết quả sẽ lưu ở", OUT_ROOT)

In [ ]:
# Ô 6: ảnh MUCCA 1.0 từ Zenodo (bản chính thức, CC-BY-4.0), kiểm md5 rồi giải nén vào đúng chỗ code của nhóm tìm.
MUCCA = ROOT / "data_external/mucca_zenodo_19231375/MUCCA_1.0"
if not (MUCCA / "images/mcac_61.jpg").exists():
    sh('wget -q -O /content/MUCCA_1.0.zip "https://zenodo.org/api/records/19231375/files/MUCCA_1.0.zip/content"')
    md5 = file_hash("/content/MUCCA_1.0.zip", "md5")
    assert md5 == "b182e445445dad017493ffb35803f816", f"md5 sai: {md5}"
    MUCCA.mkdir(parents=True, exist_ok=True)
    sh(f"unzip -q -o /content/MUCCA_1.0.zip -d {MUCCA}")
print("Số ảnh MUCCA:", len(list((MUCCA / "images").glob("*.jpg"))), "(phải là 200)")

In [ ]:
# Ô 7: checkpoint CountGD++ (1.25 GB, link trong README của CountGD++). Lần sau lấy từ Drive cho nhanh.
# Nếu gdown báo quá lượt tải: tự tải file về, upload lên Drive vào MyDrive/CS331_colab/countgd_plusplus.pth rồi chạy lại ô này.
CKPT = ROOT / "CountGDPlusPlus/checkpoints/countgd_plusplus.pth"
CACHE = OUT_ROOT / "countgd_plusplus.pth"
EXPECTED = "b89395d79711c9b9f13f3ef80cd2e528386fb1d70453f99443be4eccab33592f"  # results/countgdpp/checkpoint_audit_2026-10-14.json
if not CKPT.exists():
    if CACHE.exists():
        shutil.copy(CACHE, CKPT)
    else:
        if shutil.which("gdown") is None:
            sh("pip install -q gdown")
        sh(f'gdown "https://drive.google.com/uc?id=1j6N22TtKu2NVcKpgfrf-sJHGeLDqs9hs" -O {CKPT}')
digest = file_hash(CKPT)
assert digest == EXPECTED, f"checkpoint sha256 sai: {digest}"
if not CACHE.exists() and str(OUT_ROOT).startswith("/content/drive"):
    shutil.copy(CKPT, CACHE)
print("Checkpoint đúng bản đã audit 14/10.")
# Trọng số BERT không có trên GitHub (chỉ có config/tokenizer). Tạo bằng script gốc download_bert.py như máy Trí;
# checkpoint CountGD++ chứa đủ trọng số model (audit: missing_keys rỗng) nên giá trị BERT tải về sẽ bị ghi đè khi nạp.
BERT = ROOT / "CountGDPlusPlus/checkpoints/bert-base-uncased"
if not any((BERT / n).exists() for n in ("model.safetensors", "pytorch_model.bin")):
    sh(f"cd {ROOT / 'CountGDPlusPlus'} && {PY} download_bert.py")
print("BERT:", sorted(p.name for p in BERT.iterdir()))

In [ ]:
# Ô 8 — KIỂM A: CLIP trên Colab có cho điểm giống bản đã lưu không?
# So 30 query của p4_unseen_v1 (chỉ CHẤM LẠI khung cũ, không chỉnh gì trên tập này). Chạy bằng Python của venv.
check = '''
import json, sys
from pathlib import Path
from PIL import Image
ROOT = Path("/content/CS331"); sys.path.insert(0, str(ROOT / "experiments/countgdpp"))
from pseudo_clip_verifier import ClipVerifier
clip = ClipVerifier("cuda")
diffs = []
for f in sorted((ROOT / "results/countgdpp/p4_unseen_v1_clip_20261027").glob("*_clip_scores.json")):
    r = json.loads(f.read_text(encoding="utf-8"))
    rows = r["rows_original_order"]
    if not rows:
        continue
    with Image.open(ROOT / r["image_path"]) as im:
        image = im.convert("RGB")
    got = clip.score(image, [x["original_box_xyxy"] for x in rows], r["prompt"])
    diffs += [abs(a - x["clip_cosine_target"]) for a, x in zip(got, rows)]
print(f"KIỂM A: {len(diffs)} khung, lệch lớn nhất {max(diffs):.2e}, trung bình {sum(diffs)/len(diffs):.2e}")
'''
pathlib.Path("/content/clip_check.py").write_text(check, encoding="utf-8")
sh(f"{PY} /content/clip_check.py")
print("Đạt nếu lệch lớn nhất < 1e-3 (bản gốc chạy torch 2.9.1 / transformers 4.57.1 nên lệch rất nhỏ là bình thường).")

In [ ]:
# Ô 9 — KIỂM B + CHẠY P1 trên 5 ảnh pilot (15 query, tập dev cũ).
# --replay-b2: chạy lại B2 gốc từ cùng lượt 1 và so với B2 đã lưu; --reference-mode report: ghi độ lệch thay vì dừng.
TAUS = "0.20 0.22 0.24 0.26 0.28"
M = 10
RUN_ID = "p1_pilot_colab_" + datetime.datetime.now().strftime("%Y%m%d_%H%M")
sh(f"cd {ROOT} && {PY} experiments/countgdpp/run_b2_verified_pseudo.py"
   f" --inputs experiments/countgdpp/m4_inputs/first5_inputs.jsonl --run-id {RUN_ID} --tau-pseudo {TAUS} --m {M} --replay-b2"
   f" --b0-reference results/countgdpp/m4_first5_b0/predictions.jsonl"
   f" --b2-reference results/countgdpp/b2_two_images_20261015/predictions.jsonl results/countgdpp/b2_remaining3_20261015/predictions.jsonl"
   f" --reference-mode report")
RUN = ROOT / "results/countgdpp" / RUN_ID
shutil.copytree(RUN, OUT_ROOT / RUN_ID)
print("Đã chép kết quả sang", OUT_ROOT / RUN_ID)

In [ ]:
# Ô 10 — TÓM TẮT. Dán toàn bộ phần in ra của ô này gửi lại.
import csv
gt = {r["query_id"]: r for r in csv.DictReader(open(ROOT / "experiments/countgdpp/m4_inputs/manifest.csv", encoding="utf-8-sig"))}
def load(v):
    return [json.loads(l) for l in open(RUN / f"predictions_{v}.jsonl", encoding="utf-8")]
cfg = json.loads((RUN / "run_config.json").read_text(encoding="utf-8"))
print("run", RUN_ID, "| status", cfg["status"], "| GPU", cfg.get("device"), "| commit", subprocess.run("git -C /content/CS331 rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip())
b2 = load("B2")
s1 = [r["stage1_matches_B0"]["matches"] for r in b2]
f2 = [r["final_matches_B2_reference"]["matches"] for r in b2]
print(f"KIỂM B (so với A2000): lượt 1 khớp B0 đã lưu {sum(s1)}/{len(s1)} | B2 chạy lại khớp B2 đã lưu {sum(f2)}/{len(f2)}")
print("   (trên T4 ngày 03/11 là 1/15 và 1/15 do khác GPU; lượt 1 lệch tối đa 1 khung. Khác GPU thì xem Kiểm C ở Ô 11.)")
for r in b2:
    if not (r["stage1_matches_B0"]["matches"] and r["final_matches_B2_reference"]["matches"]):
        print("   lệch:", r["query_id"], "lượt1", r["stage1_matches_B0"], "| B2", r["final_matches_B2_reference"])
print()
print(f"{'cách':<10}{'MAE có mặt':>11}{'số query':>9}{'không mẫu':>10}{'TB số mẫu':>10}")
for v in cfg["outputs"]:
    rows = load(v)
    ok = [r for r in rows if r["execution_status"] == "ok" and gt[r["query_id"]]["presence_status"] == "present"]
    mae = sum(abs(r["final_count"] - int(gt[r["query_id"]]["gt_count"])) for r in ok) / len(ok)
    print(f"{v:<10}{mae:>11.2f}{len(ok):>9}{sum(r['no_pseudo'] for r in rows):>10}{sum(r['n_pseudo'] for r in rows)/len(rows):>10.2f}")
print()
print("Số đếm từng query (GT | B0 | B2 | các tau):")
tables = {v: {r["query_id"]: r for r in load(v)} for v in cfg["outputs"]}
for q in sorted(tables["B2"]):
    r0 = tables["B2"][q]
    print(f"  {q} {gt[q]['prompt']:<12} GT {gt[q]['gt_count']:>3} | B0 {r0['stage1_count']:>3} | " + " | ".join(f"{v} {tables[v][q]['final_count']:>3}" for v in cfg["outputs"]))

In [ ]:
# Ô 11 — KIỂM C (không cần GPU, vài giây). Cần một lần chạy T4 trước đó trên Drive để đối chiếu.
# (1) Chạy lại trên T4 có khớp TUYỆT ĐỐI lần chạy đối chiếu không (lượt 1, mẫu chọn, khung và điểm cuối, mọi cách).
# (2) Mẫu pseudo của B2 lần này có trùng mẫu B2 đã lưu (A2000, 15/10) không.
RUN1 = OUT_ROOT / "p1_pilot_colab_20261103_1519"  # lần chạy T4 đầu tiên (03/11); đổi tên nếu muốn so với lần khác
RUN2 = OUT_ROOT / RUN_ID
assert RUN2 != RUN1, "RUN_ID đang trùng lần đối chiếu: chạy lại Ô 9 trước"
def jl(path):
    return {r["query_id"]: r for r in map(json.loads, open(path, encoding="utf-8"))}
keys = ("stage1_boxes", "stage1_scores", "pseudo_proposal_ids", "final_boxes", "final_scores")
for v in json.loads((RUN2 / "run_config.json").read_text(encoding="utf-8"))["outputs"]:
    r1, r2 = jl(RUN1 / f"predictions_{v}.jsonl"), jl(RUN2 / f"predictions_{v}.jsonl")
    bad = sorted(q for q in r1 if any(r1[q][k] != r2[q][k] for k in keys))
    print(f"KIỂM C {v:<9}: lần này khớp {RUN1.name} tuyệt đối {len(r1) - len(bad)}/{len(r1)}" + (f" | lệch: {bad}" if bad else ""))
print()
saved = {}
for f in ("b2_two_images_20261015", "b2_remaining3_20261015"):
    saved.update(jl(ROOT / "results/countgdpp" / f / "predictions.jsonl"))
t4 = jl(RUN2 / "predictions_B2.jsonl")
print("Mẫu pseudo B2: lần này so với A2000 (id = số thứ tự đề xuất của CountGD++)")
for q in sorted(t4):
    a, b = t4[q]["pseudo_proposal_ids"], saved[q]["pseudo_proposal_ids"]
    tag = "giống" if sorted(a) == sorted(b) else "KHÁC"
    print(f"  {q}: {tag:<5} lần này {a} | A2000 {b} | B2 lần này {t4[q]['final_count']} vs A2000 {saved[q]['final_count']}")

In [ ]:
# Ô 12 — BẢNG ĐỐI CHỨNG (kế hoạch Trí §3.5), cùng bộ lọc cuối đã khóa 21/10, không chạy lại CountGD++.
# Mỗi nguồn khung X (B0 = lượt 1, B2, từng tau) có 3 cách: X (giữ hết), X+S (lọc điểm CountGD++), X+F (Fusion 50/50).
# Tên trong kế hoạch: F = B2+F, S-only = B2+S, C6 = B0+F, P+F = tau+F. CLIP chấm lại mọi khung (vài phút trên T4).
# Phiên mới: chạy Ô 1–5 rồi chạy thẳng ô này; SRC mặc định là lần chạy T4 đầu tiên trên Drive.
# LƯU Ý pilot: ngưỡng S và F được khớp trên chính 5 ảnh này (21/10), nên F/S-only trông tốt hơn thực tế. Chỉ để kiểm code.
SRC = RUN_ID if "RUN_ID" in globals() else "p1_pilot_colab_20261103_1519"
run_dir = ROOT / "results/countgdpp" / SRC
if not run_dir.exists():  # phiên mới: chỉ chép từ Drive những file cần dùng
    run_dir.mkdir(parents=True)
    for f in [OUT_ROOT / SRC / "run_config.json", *sorted((OUT_ROOT / SRC).glob("predictions_*.jsonl"))]:
        shutil.copy2(f, run_dir / f.name)
CTRL = SRC + "_controls"
sh(f"cd {ROOT} && {PY} experiments/countgdpp/p1_controls.py --run results/countgdpp/{SRC} --run-id {CTRL}")
sh(f"cd {ROOT} && {PY} evaluation/evaluate_research.py --controls results/countgdpp/{CTRL}"
   f" --manifest experiments/countgdpp/m4_inputs/manifest.csv --run-id {CTRL}_eval")
for d in (CTRL + "_clip", CTRL, CTRL + "_eval"):
    shutil.copytree(ROOT / "results/countgdpp" / d, OUT_ROOT / d)
print("Đã chép sang Drive:", OUT_ROOT / (CTRL + "_clip"), OUT_ROOT / CTRL, OUT_ROOT / (CTRL + "_eval"))

## Sau khi chạy xong

1. Dán phần in ra của **Ô 8**, **Ô 10**, **Ô 11** và **Ô 12** gửi lại.
2. **Kiểm B** chỉ khớp tuyệt đối khi chạy cùng loại GPU với bản đã lưu (A2000). Trên GPU khác thì dựa vào **Kiểm C**: chưa đạt 15/15 ở mọi cách thì **chưa được dùng số P1**. Tiêu chí chính thức chờ nhóm chốt.
3. Kết quả đầy đủ nằm trên Drive: `MyDrive/CS331_colab/<RUN_ID>/` (predictions từng cách, `candidates.jsonl` ghi điểm CLIP của từng ứng viên, ảnh `candidates.jpg` và `final_*.jpg` mỗi query).
4. Đây mới là **pilot 5 ảnh**. Chọn `tau_pseudo` chính thức phải đợi nhóm khóa tập dev 30 ảnh.